# Initial Set-Up

## Imports

In [ ]:
%load_ext autoreload
%autoreload 2
import json
import os 
import sys

sys.path.append("../src")

import matplotlib as mpl
import matplotlib.pyplot as plot
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
import matplotlib.ticker as ticker
from matplotlib.colors import LinearSegmentedColormap, to_rgb

import pandas as pd
import numpy as np
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_regression
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

from collections import Counter, defaultdict
from tqdm import tqdm
import pickle

from bisect import bisect_left
from global_utils.graphs_utils import prepare_subplots, color_dic, arrange_bar_plots, arrange_twin_plots, prepare_subplots_big, fancy_histogram

## Read Files

In [ ]:
DATA_DIR = "../Database/data"
fontsize = 18
SAVE_PATH_MANUAL = "../../Images"
plot.rcParams['savefig.format'] = "svg"
plot.rcParams['figure.figsize'] = [6.6, 5.0]

In [ ]:
meter_values = pd.read_csv(os.path.join(DATA_DIR, '_MonitoringChargerMeterValues__202512191557.csv'))
meter_values['MeterValueTimeStamp'] = pd.to_datetime(meter_values['MeterValueTimeStamp'])
meter_values['UpdatedTimeStamp'] = pd.to_datetime(meter_values['UpdatedTimeStamp'])

In [ ]:
transactions = pd.read_csv(
    os.path.join(DATA_DIR, '_TransactionsAndReservationsSystemTransactions_v2__202512191543.csv'))
transactions['StartAt'] = pd.to_datetime(transactions['StartAt'])
transactions['EndAt'] = pd.to_datetime(transactions['EndAt'])
name_of_chargers = list(Counter(transactions["ChargerID"]))

In [ ]:
filenames = ["eventshistory_01DecTo15Dec.csv", "eventshistory_15DecTillDate.csv"]
dataframes = []

for filename in filenames:
    dataframes.append(pd.read_csv(os.path.join(DATA_DIR,filename)))
events = pd.concat(dataframes, ignore_index=True)
events['event_time'] = pd.to_datetime(events['event_time'])
events['event_update_time'] = pd.to_datetime(events['event_update_time'])
events['event_time'] = events['event_time'].dt.tz_localize('UTC').dt.tz_convert('UTC+05:30')
events['event_update_time'] = events['event_update_time'].dt.tz_localize('UTC').dt.tz_convert('UTC+05:30')
events['EvseID'] = None
events['ConnectorID'] = None

In [ ]:
charger_events = defaultdict(list)
for _, event in tqdm(events.iterrows(), total=len(events)):
    source_split = event['source'].split("/") + ["","","",""]       
    event['EvseID'] = int(source_split[1]) if source_split[0] == "EVSE" else None  
    event['ConnectorID'] = int(source_split[3]) if source_split[2] == "Connector" else None  
    charger_events[event['device_id']].append(event)
charger_transactions = defaultdict(list)
for _, transaction in tqdm(transactions.iterrows(), total=len(transactions)):
    charger_transactions[transaction['ChargerID']].append(transaction)
charger_meters = defaultdict(dict)
for _, meter in tqdm(meter_values.iterrows(), total=len(meter_values)):
    if meter['TransactionID'] not in charger_meters[meter['ChargerID']]:
        charger_meters[meter['ChargerID']][meter['TransactionID']] = []
    charger_meters[meter['ChargerID']][meter['TransactionID']].append(meter)

In [ ]:
all_transaction_info = {}
tolerance = pd.to_timedelta('2m')


for name in name_of_chargers:
    all_transaction_info[name] = []

for charger in tqdm(charger_transactions):
    device_events = sorted(charger_events[charger], key=lambda x: x['event_time'])
    charger_meter_values = charger_meters[charger]
    for transaction in sorted(charger_transactions[charger], key=lambda x: x['StartAt']):
        start_at = transaction['StartAt']
        idx_event = bisect_left(device_events, start_at - tolerance, key=lambda x: x['event_time'])
        end_at = transaction['EndAt']

        evseid_transaction, connectorid_transaction = transaction['EvseID'], transaction['ConnectorID']

        transaction_events = []
        while idx_event < len(device_events) and device_events[idx_event]['event_time'] <= end_at+tolerance:

            evseid_event = device_events[idx_event]['EvseID']
            connectorid_event = device_events[idx_event]['ConnectorID']

            if evseid_transaction == evseid_event or evseid_event is None:
                if connectorid_transaction == connectorid_event or connectorid_event is None:
                    transaction_events.append(device_events[idx_event]) # Only assign the event to the transaction if the event's EVSE ID and connector ID match the transaction's, or if we don't have information about the event's.

            idx_event += 1
        # all_transaction_info[transaction['ChargerTransactionID']] = {'start_at': start_at, 'end_at': end_at, 'events': transaction_events, 'meter_values': charger_meter_values.get(transaction['ChargerTransactionID'], [])}
        all_transaction_info[transaction['ChargerID']].append({'transaction': transaction, 'events':transaction_events, 'meter_values':charger_meter_values.get(transaction['ChargerTransactionID'], [])})
    

In [ ]:
combine_error_dic = {}
for name in (events['name'].unique()):
    if 'outlet' in name or "allOutletsUnavailable" in name:
        combine_error_dic[name] = "outletX"
    elif 'Outlet_DC_codes' in name:
        combine_error_dic[name] = "Outlet_DC_codes (X)"
    elif 'PowerConverter' in name:
        combine_error_dic[name] = 'PowerConverter (X)' 
    elif 'IMD_BC' in name:
        combine_error_dic[name] = 'IMD_BCX'
    elif 'Converter_Group_DC' in name:
        combine_error_dic[name] = 'Converter_Group_DCX' 
    elif 'StateMonitor' in name:
        combine_error_dic[name] = 'StateMonitor_X' 
    elif 'DC_Contactor' in name:
        combine_error_dic[name] = 'DC_Contactor_X' 
    elif 'CableTempSensor' in name:
        combine_error_dic[name] = 'DCX_CableTempSensor'
    else:
        combine_error_dic[name] = name

combine_error_dic['None'] = 'None'
combine_error_dic['Error_Event'] = "Error_Event"

## Creating Functions

In [ ]:
def count_number_of_events(event_info,  
                           event_name_dic={},
                           pass_dic=None,
                           INCLUDE_VALUE=set(),
                           INCLUDE_TYPES={"Error": "Error_Event"},
                           INCLUDE_NONE="None"):
    
    assert isinstance(INCLUDE_TYPES, dict)
    assert isinstance(INCLUDE_VALUE, set)

    if pass_dic is None:
        pass_dic = defaultdict(lambda: 0)
        
    if len(event_info):
        for event in event_info:
            if INCLUDE_TYPES:
                event_type = event['type']
                if event_type in INCLUDE_TYPES:
                    pass_dic[INCLUDE_TYPES[event_type]] += 1

            event_name = event['name']
            event_name = event_name_dic.get(event_name, event_name)
            if event_name in INCLUDE_VALUE:
                event_name = event_name + "_" + event['value']
            
            pass_dic[event_name] += 1
            
    elif INCLUDE_NONE:
        pass_dic[INCLUDE_NONE] += 1
    
    return pass_dic

In [ ]:
def make_events_plot(fig, ax, array, statistics, show_only, bins, diff, total_metered_transactions, loc=(1,0.74)):

    patches = []
    for i in range(show_only):
        event_name =list(statistics)[i] 
        hist = np.histogram(array[event_name], bins=bins)
        hist_0 = hist[0]/sum(hist[0]) * 100
        ax.bar(hist[1][:-1], hist_0, width=diff, align='edge',  alpha=0.1)
        ax.step(np.concat([[bins[0]-diff], hist[1]]), np.concat([[0], hist_0, [0]]), where="post")
        patches.append( 
            mpatches.Patch(facecolor=color_dic['default'][i], label=event_name + " | " +"{:.3f}".format(statistics[event_name][0]/total_metered_transactions), edgecolor="k")
        )


    leg = ax.legend(title="Event name | Relative frequency:", handles=patches)
    prepare_subplots([ax])
    ax.add_artist(leg)
    for i in range(show_only):
        event_name =list(statistics)[i] 
        ax.axvline(statistics[event_name][1], color=color_dic['default'][i], label="{:.2f} seconds".format(statistics[event_name][1]), linestyle="--")
    ax.legend(title="Mean time difference", bbox_transform=ax.transAxes, bbox_to_anchor=loc)

In [ ]:
def fancy_plot(ax, xs, ys, event_name, xlabel="# Transactions logged", ylabel="Event frequency", color=color_dic['default'][0], marker="o"):


    q75, q25 = np.percentile(ys, [75,25])
    max_range = q75 + (q75-q25)*3

    ax.axhline(max_range, color="k", linestyle="--")
    xmin, xmax = ax.get_xlim()
    ymin, ymax = ax.get_ylim()
    ax.fill_between([0,1e4], 0, max_range, color="g", alpha=0.1)
    ax.fill_between([0,1e4], max_range, 100, color="r", alpha=0.1)

    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)

In [ ]:
def fancy_histogram(fig, ax, array, bins, weights=None, color = color_dic['default'][0],linestyle="-", linewidth = 2, alpha=0.1):
    if weights is None:
        weights = np.full(len(array),100)/(1e-12+len(array))
    # ax.hist(array, bins=bins, color=color, alpha=alpha, weights=weights,edgecolor=color)
    counts, bins,_ = ax.hist(array, bins=bins, color=color, weights=weights, histtype="step",linestyle=linestyle,linewidth=linewidth)
    bins = [bins[0]] +  [ element for i in range(1, len(bins)-1) for element in [bins[i], bins[i]]  ] + [bins[-1]]
    counts = [element for i in range(len(counts)) for element in [counts[i], counts[i]]]
    ax.fill_between(bins, [0]*len(bins), counts, alpha=alpha, color=color)

    # hist = np.histogram(array, bins=bins)
    # hist_0 = hist[0]/sum(hist[0]+1e-12)*100
    # diff = bins[1:]-bins[:-1]
    # ax.bar(hist[1][:-1], hist_0, width=diff, align='edge', alpha=0.1, color=color)
    # ax.step(np.concat([[bins[0]-diff[0]], hist[1]]), np.concat([[0], hist_0, [0]]), where="post", color=color, linestyle=linestyle)

# Processing Data

## Meter Values and Events

In [ ]:
threshold_time = pd.Timedelta('5m')
min_threshold_energy = 1 #*10^3
max_threshold_energy = 1e3 #*10^3
energy_delivered_for_filtered_transactions = []
energy_delivered_info = []
time_when_no_meter_values = []
errors_per_transaction = [[], []]


types_of_transactions = defaultdict(lambda: [0,0])
durations_of_transactions = defaultdict(lambda: [0,0])
types_of_errors = [defaultdict(lambda: 0), defaultdict(lambda: 0)]

threshold_event_time = pd.Timedelta('5m').seconds
events_near_start = defaultdict(list)
events_near_end = defaultdict(list)
total_metered_transactions = 0
temp = []

total_energy_delivered = []
nan_energy_delivered = 0

errors_per_charger = defaultdict(lambda: [defaultdict(lambda: 0), defaultdict(lambda: 0)])
for name in name_of_chargers:
    for transaction in all_transaction_info[name]:


        transaction_info = transaction['transaction']
        initial_SoC, end_SoC = float(transaction_info['InitialSoC']), transaction_info['EndSoC']
        start_at, end_at = transaction_info['StartAt'], transaction_info['EndAt']
        time_of_transaction = end_at-start_at
        energy_delivered = transaction_info['EnergyDelivered']/1e3
        end_result = transaction_info['EndResult']

        event_info = transaction['events']
        meter_values_info = transaction['meter_values']
        # Unfiltered transactions

        index = 0 if end_result == "EVDisconnected" else 1
        count_number_of_events(event_info, combine_error_dic, types_of_errors[index], set())
        count_number_of_events(event_info, combine_error_dic, errors_per_charger[name][index], set())
        types_of_transactions[name][index] += 1

        durations_of_transactions[name][index] += time_of_transaction.total_seconds()/ (60*60) # hours

        if (pd.isnull(energy_delivered)):
            nan_energy_delivered += 1
        else:
            total_energy_delivered.append(energy_delivered)
        # Filter transactions with null, or energy delivered excedding thresholds
        if ( pd.isnull(energy_delivered) or energy_delivered < min_threshold_energy or energy_delivered >= max_threshold_energy):
            continue


        # Filter transactions will null start or end time 
        #   Futher filter transactions based on the total time
        if (pd.isnull(end_at) or pd.isnull(start_at)) or \
           (end_at - start_at) < threshold_time:
            continue
        
        
        # Making the graphs about the errors
        energy_delivered_for_filtered_transactions.append(energy_delivered)
        transaction_duration = (end_at-start_at).seconds/(60*60) # Hours
        temp_error_dic = count_number_of_events(event_info, combine_error_dic)

        energy_delivered_info.append([energy_delivered, end_result, temp_error_dic, transaction_duration])
        


        # Start with meter case 
        initial_charging_time_guess, end_charging_time_guess = None, None
        if len(meter_values_info):
            # When we have this information
            base_energy_deliver = float(json.loads(meter_values_info[0]["RawData"])['EnergyDelivered'])
            last_energy_deliver = base_energy_deliver
            for value in meter_values_info:
                timestamp = value["MeterValueTimeStamp"]
                RawData   = json.loads(value["RawData"])
                guess_energy = float(RawData['EnergyDelivered'])
                if (guess_energy-base_energy_deliver == 0):
                    # Save the latest moment where the energy delivered remains constant
                    #   Equals sign is to get the latest update possible (charger may be connected without energy delivery)
                    initial_charging_time_guess = timestamp
                
                elif (guess_energy>last_energy_deliver):
                    # Save the earliest moment when the energy delievered remains constant 
                    #   Stops in the first instance where the guess_energy reaches the final value 
                    end_charging_time_guess = timestamp
                last_energy_deliver = guess_energy
            
            if initial_charging_time_guess is None or end_charging_time_guess is None:
                continue
            
            if len(event_info):
                for event in event_info:
                    event_name = combine_error_dic[event['name']]
                    if event_name == "Color" or event_name == "State":
                        event_name = event_name + "_" + event['value']
                    event_time_stamp = event['event_time']
                    diff_to_guess_start = ((event_time_stamp - initial_charging_time_guess).seconds)
                    diff_to_guess_end = ((event_time_stamp - end_charging_time_guess).seconds)
                    
                    if np.abs(diff_to_guess_start) < threshold_event_time:
                        events_near_start[event_name].append(diff_to_guess_start)
                    elif np.abs(diff_to_guess_end) < threshold_event_time:
                        events_near_end[event_name].append(diff_to_guess_end)

            total_metered_transactions += 1

        else:
            pass
    

energy_delivered_array = np.array([element[0] for element in energy_delivered_info])
total_energy_delivered = np.array(total_energy_delivered)

In [ ]:
bins = np.arange(0,605,10)
energy_dictionary = np.array([ key for key in bins ])
time_of_transaction_vs_energy_deliveres = []
for _ in range(len(energy_dictionary)-1):
    time_of_transaction_vs_energy_deliveres.append([])

for element in energy_delivered_info:
    index = 0
    while bins[index+1] < element[0] and index < len(bins)-2:
        index += 1
    time_of_transaction_vs_energy_deliveres[index].append(element[3])


In [ ]:
events_near_start_statistic = {key: [len(value), np.mean(value), np.std(value)] for key, value in events_near_start.items()}
events_near_start_statistic = {key: value + [value[0]/total_metered_transactions*( 1/(value[1]+1e-5) + value[2]/(value[1]*value[1]+1e-5))] for key, value in events_near_start_statistic.items()}
events_near_end_statistic = {key: [len(value), np.mean(value), np.std(value)] for key, value in events_near_end.items()}
events_near_end_statistic = {key: value + [value[0]/total_metered_transactions*( 1/(value[1]+1e-5) + value[2]/(value[1]*value[1]+1e-5))] for key, value in events_near_end_statistic.items()}

events_near_start_statistic = dict(sorted(events_near_start_statistic.items(), key=lambda x: x[1][3], reverse=True))
events_near_end_statistic = dict(sorted(events_near_end_statistic.items(), key=lambda x: x[1][3], reverse=True))

In [ ]:
keys = set(types_of_errors[0].keys()).union( types_of_errors[1].keys() )
prob_fault_if_event = {}


total_number_events = np.sum( [types_of_errors[0][key]+types_of_errors[1][key] for key in keys if not (key == "Error_Event" or key =="None")] )
total_number_events_when_fault = np.sum( [types_of_errors[1][key] for key in keys if not (key == "Error_Event" or key =="None")] )
prob_fault = total_number_events_when_fault/total_number_events

# Compute the probability of event resulting in fault
for key in keys:
    if (key == "None"):
        continue
    prob_event = (types_of_errors[0][key]+types_of_errors[1][key])/total_number_events 
    prob_event_when_fault = types_of_errors[1][key]/(total_number_events_when_fault + 1e-12)
    prob_fault_if_event[key] = (prob_event_when_fault*prob_fault)/(prob_event + 1e-12)

    if prob_fault_if_event[key] - prob_fault_if_event[key] != 0:
        print(key)

prob_fault_if_event = dict(sorted(prob_fault_if_event.items(), key=lambda x: x[1], reverse=True))

In [ ]:
keys = set(types_of_errors[0].keys()).union( types_of_errors[1].keys() )
freq_errors_per_charger = {
    key : [(errors_per_charger[name][0][key]+errors_per_charger[name][1][key]) / (1e-12+types_of_transactions[name][0]+types_of_transactions[name][1]) for name in name_of_chargers]
    for key in keys
}

unhealthy_chargers = defaultdict(list)
for key in keys:
    q75, q25 = np.percentile(freq_errors_per_charger[key], [75,25])
    max_range = q75 + (q75-q25)*3
    for index_name, name in enumerate(name_of_chargers):
        if freq_errors_per_charger[key][index_name] > max_range:
            unhealthy_chargers[name].append([key, freq_errors_per_charger[key][index_name]])

In [ ]:
# Quick percentages on energy delivered
total_number = len(total_energy_delivered) + nan_energy_delivered


print(" \'nan\' energy delivered: ", nan_energy_delivered/total_number*100)
print(" Negative energy delivered: ", len(total_energy_delivered[[element < 0 for element in total_energy_delivered]])/total_number*100)
print(" High energy delivered: ", len(total_energy_delivered[[element > 1000 for element in total_energy_delivered]])/total_number*100)
print(" Small energy delivered: ", len(total_energy_delivered[[1> element >= 0 for element in total_energy_delivered]])/total_number*100)
print(" Zero energy delivered: ", len(total_energy_delivered[[element == 0 for element in total_energy_delivered]])/total_number*100)

## Charging Times

In [ ]:
threshold_time = pd.Timedelta('5m')
min_threshold_energy = 1 #*10^3
max_threshold_energy = 1e3 #*10^3
energy_delivered_per_charging_times = []

energy_delivered_per_charger = defaultdict(list)
charging_duration_per_charger = defaultdict(list)

event_threshold_time = (threshold_time.seconds)/2
start_charging_events = ["VehicleId", "AuthorizationAttempt"]
end_charging_events =["State_B1", "State_C2", "State_B2", "Color_GREEN (AVAILABLE)"]

index = 0
for name in name_of_chargers:
    for transaction in all_transaction_info[name]:
        temp_error_dic = defaultdict(lambda: 0)
        

        transaction_info = transaction['transaction']
        initial_SoC, end_SoC = float(transaction_info['InitialSoC']), transaction_info['EndSoC']
        start_at, end_at = transaction_info['StartAt'], transaction_info['EndAt']
        energy_delivered = transaction_info['EnergyDelivered']/1e3
        end_result = transaction_info['EndResult']

        event_info = transaction['events']
        meter_values_info = transaction['meter_values']

        # if transaction_info['ChargerTransactionID'] != "910dd5ef-7ddb-419c-a73f-c386d2d249d9":
            # continue

        initial_charging_time_guess, end_charging_time_guess = None, None

        # Filter transactions with null, or energy delivered excedding thresholds
        if ( pd.isnull(energy_delivered) or energy_delivered < min_threshold_energy or energy_delivered >= max_threshold_energy):
            continue


        # Filter transactions will null start or end time and based on the total time
        if (pd.isnull(end_at) or pd.isnull(start_at)) \
            or (end_at - start_at) < threshold_time:
            continue
        index += 1

        initial_event_time_guess = {key: None for key in start_charging_events}
        end_event_time_guess = {key: None for key in end_charging_events}
        

        if len(event_info):
            for event in event_info:
                event_name = combine_error_dic[event['name']] 
                temp_error_dic[event_name] += 1
                if event['type'] == 'Error':
                    temp_error_dic['Error_Event'] += 1
                
                if (event_name == "Color" or event_name == "State"):
                    event_name = event_name + "_" + event['value']
                
                if event_name in start_charging_events:
                    event_time = event['event_time']
                    if initial_event_time_guess[event_name] is None:
                        initial_event_time_guess[event_name] = event['event_time']
                elif event_name in end_charging_events:
                    event_time = event['event_time']
                    if np.abs(event_time-start_at).seconds < event_threshold_time:
                        continue
                    if end_event_time_guess[event_name] is None:
                        end_event_time_guess[event_name] = event['event_time']
        else:
            temp_error_dic['None'] += 1
        
        second_initial_time_guess = None
        second_end_time_guess = None
        for value in initial_event_time_guess.values():
            if value is None:
                continue
            if second_initial_time_guess is None:
                second_initial_time_guess = value
            second_initial_time_guess = value if value < second_initial_time_guess else second_initial_time_guess

        for value in end_event_time_guess.values():
            if value is None:
                continue
            if second_end_time_guess is None:
                second_end_time_guess = value
            second_end_time_guess = value if value > second_end_time_guess else second_end_time_guess

        second_initial_time_guess = start_at if second_initial_time_guess is None else second_initial_time_guess
        second_end_time_guess = end_at if second_end_time_guess is None else second_end_time_guess


        if len(meter_values_info) >= 2:
            current_initial_SoC_guess, current_end_SoC_guess = 1000, 1000
            base_energy_deliver = float(json.loads(meter_values_info[0]["RawData"])['EnergyDelivered'])
            last_energy_deliver = float(json.loads(meter_values_info[0]["RawData"])['EnergyDelivered'])
            for value in meter_values_info:
                timestamp = value['MeterValueTimeStamp']
                RawData   = json.loads(value["RawData"])
                guess_soc = float(RawData['SoC'])
                guess_energy = float(RawData['EnergyDelivered'])
                if (guess_energy-base_energy_deliver == 0):

                # if np.abs(guess_soc-initial_SoC) <= np.abs(current_initial_SoC_guess-initial_SoC):
                    # The current SoC value is closer to the Initial_SoC from transaction information
                    #   Equals sign is to get the latest update possible (charger may be connected without energy delivery)
                    current_initial_SoC_guess  = guess_soc
                    initial_charging_time_guess = timestamp
                
                elif (guess_energy>last_energy_deliver):
                # elif np.abs(guess_soc-end_SoC) < np.abs(current_end_SoC_guess-end_SoC):
                    # The current SoC value is closer to the End_SoC from transaction information
                    #   Stops in the first instance where the guess_soc reaches the final value 
                    current_end_SoC_guess   = guess_soc
                    end_charging_time_guess = timestamp
                last_energy_deliver = guess_energy
        
            
        initial_charging_time_guess = second_initial_time_guess if initial_charging_time_guess is None else initial_charging_time_guess
        end_charging_time_guess = second_end_time_guess if end_charging_time_guess is None else end_charging_time_guess

        if initial_charging_time_guess > end_charging_time_guess:
            initial_charging_time_guess = start_at
            end_charging_time_guess = end_at
        if not(initial_charging_time_guess is None) and not(end_charging_time_guess is None):
            
            diff = (end_charging_time_guess - initial_charging_time_guess).total_seconds() /(60*60) # Hours
            diff_naive = (end_at - start_at).total_seconds() / (60*60) # Hours
            energy_delivered_per_charging_times.append([energy_delivered, diff, end_result, temp_error_dic, diff_naive])
            energy_delivered_per_charger[name].append(energy_delivered)
            charging_duration_per_charger[name].append(diff)

In [ ]:
X = np.array([[element[3][key] for key in set(combine_error_dic.values())] for element in energy_delivered_per_charging_times])
y = np.array([element[0]/(1e-6+element[1]) for element in energy_delivered_per_charging_times])
charging_rate_correlations = {}
for index, key in enumerate(set(combine_error_dic.values())):

    if np.std(X[:,index]) != 0:
        charging_rate_correlations[index] = np.corrcoef(X[:,index], y)[0,1]
    

charging_rate_correlations = dict(sorted(charging_rate_correlations.items(), key= lambda x: x[1], reverse=True))

## Events outside transactions

In [ ]:
threshold_time = pd.Timedelta('5m')
min_threshold_energy = 1 #*10^3
max_threshold_energy = 1e3 #*10^3

events_outside_transactions = defaultdict(lambda: defaultdict(lambda: 0))
diff_between_events = []

for name in name_of_chargers:
    events_caught_by_transactions = []
    for transaction in all_transaction_info[name]:
        event_info = transaction['events']
        # event_info = [(event['device_event_id'], event['event_time']) for event in event_info]
        event_info = [event['device_event_id'] for event in event_info]
        events_caught_by_transactions = events_caught_by_transactions + event_info
    
    device_events = charger_events[name]
    count_matches = 0
    events_caught_by_transactions = set(events_caught_by_transactions)
    diff_between_events.append(len(device_events)-len(events_caught_by_transactions))
    
    for event in device_events:
        # test = (event['device_event_id'], event['event_time'])
        test = event['device_event_id']
        if not test in events_caught_by_transactions:
            event_name = combine_error_dic[event['name']]
            events_outside_transactions[name][event_name] += 1
            event_type = event['type']
            if event_type == "Error":
                events_outside_transactions[name]["Error_Event"] += 1
        else:
            count_matches += 1
    
    
        

## Data Information

In [ ]:
number_events_per_transaction = np.sum([ len(transaction['events'])for name in name_of_chargers for transaction in all_transaction_info[name]  ])
total_number_of_transactions = np.sum([np.sum(types_of_transactions[name]) for name in name_of_chargers])
number_meter_points_per_transaction = np.sum([ len(transaction['meter_values']) for name in name_of_chargers for transaction in all_transaction_info[name]  ])

print(len(events))
print(number_events_per_transaction)
print((types_of_errors[0]['None']+types_of_errors[1]['None'])/total_number_of_transactions)

print(total_number_of_transactions)

print(number_meter_points_per_transaction)

## Gap Information

In [ ]:
for specific_charger in name_of_chargers:
    specific_transactions = charger_transactions[specific_charger].copy()
    specific_transactions = specific_transactions[["ChargerTransactionID", "ChargerID", "StartAt","EndAt","EnergyDelivered"]]
    specific_transactions.sort_values("StartAt", inplace=True)
    mask = specific_transactions["EndAt"].isna() 
    specific_transactions.loc[mask, "EndAt"] = specific_transactions.loc[mask, "StartAt"]+pd.Timedelta(seconds=1)
    data_gaps = get_data_gaps_df(specific_transactions)
    if not data_gaps.empty:
        chargers_with_gaps.append(specific_charger)

        all_gaps = pd.concat([all_gaps, data_gaps])

all_gaps = all_gaps.sort_values("StartAt")

usable_time_ranges = {}
specific_charger = name_of_chargers[4]
for specific_charger in name_of_chargers:
    specific_gaps = all_gaps[all_gaps["ChargerID"] == specific_charger]
    min_time = charger_transactions[specific_charger]["StartAt"].min()
    max_time = charger_transactions[specific_charger]["StartAt"].max()
    next_start = min_time
    temp = []
    for _, row in specific_gaps.iterrows():
        temp.append([next_start, row["EndAt"]])
        next_start = row["Next StartAt"]
    
    temp.append([next_start, max_time])
    usable_time_ranges[specific_charger] = temp

starts = [(el, 1) for el in all_gaps["StartAt"].to_list()]
ends   = [(el, -1) for el in all_gaps["Next StartAt"].to_list()]
combine = starts + ends
combine = sorted(combine, key=lambda x: x[0])
steps = [(global_min_temp,0)]
for el in combine:
    steps.append([el[0], steps[-1][1]+el[1]])

for el in steps:
    if len(el)!=2:
        print(el)

steps.append([global_max_temp, steps[-1][1]])
steps = np.array(steps)

In [ ]:
key_t = {}
offline_transactions = {}


for specific_charger in tqdm(name_of_chargers):

    specific_transactions = charger_transactions[specific_charger]
    starts = np.array(specific_transactions["StartAt"].tolist())
    min_time = starts.min()
    max_time = starts.max()

    specific_events = charger_events[specific_charger]
    specific_events = specific_events[(specific_events["name"]=="Connectivity Update") & (specific_events["event_time"] >= min_time) & (specific_events["event_time"] <= max_time)].sort_values("event_time")

    updates_matrix = specific_events[["event_time", "value"]].values

    updates = []
    prev_start = updates_matrix[0][0]
    prev_value = updates_matrix[0][1]

    for start, value in updates_matrix[1:]:
        updates.append([prev_start, start, prev_value])
        prev_start = start
        prev_value = value
    updates.append([prev_start, max_time, prev_value])
    mask = pd.arrays.IntervalArray.from_tuples(
        [( st, ed ) for st, ed, vl in updates if vl == "Disconnected"],
        closed = "both"
    )
    transactions_in_disconnects = specific_transactions[ specific_transactions["StartAt"].apply(lambda t: any(t in iv for iv in mask))]

    total_transactions = specific_transactions.shape[0]
    nmr_transactions_in_disconnects = transactions_in_disconnects.shape[0]

    offline_key_in_disconnects = np.array(transactions_in_disconnects["Offline"].tolist())
    total_offline_key_t = specific_transactions[specific_transactions["Offline"]=="t"]
    total_offline_key_t = total_offline_key_t.shape[0]

    key_t[specific_charger] = ([offline_key_in_disconnects, total_offline_key_t])
    offline_transactions[specific_charger] = ([total_transactions, nmr_transactions_in_disconnects])

In [ ]:
frac_in_disconnect = [ dis/tot for tot,dis in offline_transactions.values()]

In [ ]:
distance_to_min = []
distance_to_max = []

specific_charger = "77moda"
for specific_charger in tqdm(name_of_chargers):
    specific_transactions = charger_transactions[specific_charger]
    starts = np.array(specific_transactions["StartAt"].tolist())
    min_time = starts.min()
    max_time = starts.max()

    specific_events = charger_events[specific_charger]
    specific_events = specific_events[(specific_events["name"]=="Connectivity Update") & (specific_events["event_time"] >= min_time) & (specific_events["event_time"] <= max_time)].sort_values("event_time")

    # display(specific_events[specific_events["event_time"]>pd.Timestamp("2025-12-09").tz_localize("UTC")])

    updates_matrix = specific_events[["event_time", "value"]].values

    updates = []
    prev_start = updates_matrix[0][0]
    prev_value = updates_matrix[0][1]

    for start, value in updates_matrix[1:]:
        updates.append([prev_start, start, prev_value])
        prev_start = start
        prev_value = value
    updates.append([prev_start, max_time, prev_value])
    disconnect_updates = [(st, ed) for st,ed,val in updates if val == "Disconnected"]
    specific_gaps = all_gaps[all_gaps["ChargerID"] == specific_charger]
    specific_gaps = [(st, ed) for st, ed in specific_gaps[["EndAt", "Next StartAt"]].values.tolist()]

    for gap in disconnect_updates:
    
        duration = gap[1]-gap[0]
        if duration < pd.Timedelta('5D'):
            continue

        if len(specific_gaps) == 0:
            continue
        index = np.argmin( [np.abs(gap[0]- st) for st, ed in specific_gaps] )

        distance_to_min.append((gap[0] - specific_gaps[index][0]).total_seconds()/(24*3600))
        distance_to_max.append((gap[1] - specific_gaps[index][1]).total_seconds()/(24*3600))
    


# Graphs

## Overview

In [ ]:
# Number of events per transaction
fig, ax = plot.subplots()
fig2, ax2 = plot.subplots()
fig3, ax3 = plot.subplots()
axs = [ax, ax2, ax3]
figs = [fig, fig2, fig3]
colors = color_dic["siemens_colors"]
prepare_subplots_big(axs,key="siemens_colors_graphs", background_color=colors[0], tick_color=colors[1], grid_color=colors[2])
for f in figs:
    f.set_facecolor("none")

edgecolor = np.array(color_dic['siemens_colors_graphs'])[9]

SHOW = False
SAVE = False

number_events_per_transaction = [ len(transaction['events'])/1000 for name in name_of_chargers for transaction in all_transaction_info[name]  ]

number_meter_points_per_transaction = [ len(transaction['meter_values']) for name in name_of_chargers for transaction in all_transaction_info[name]  ]

duration_of_transactions = []
null_information = 0
for name in name_of_chargers:
    for transaction in all_transaction_info[name]:

        start_at = transaction['transaction']['StartAt']
        end_at = transaction['transaction']['EndAt']

        if (pd.isnull(start_at) or pd.isnull(end_at)):
            null_information += 1
            continue

        duration_of_transactions.append( (end_at-start_at).total_seconds()/(60*60)) # minutes


diff = 2.5e-1
bins = np.arange(0, np.max(number_events_per_transaction)+diff/2, diff)
# fancy_histogram(fig,ax,number_events_per_transaction,bins,linestyle="-", color=color_dic["siemens_colors_graphs"][0],weights=np.ones(len(number_events_per_transaction)))
ax.hist(number_events_per_transaction, bins=bins, edgecolor=edgecolor)

diff = 50
bins = np.arange(0, np.max(number_meter_points_per_transaction)+diff/2, diff)
print(np.max(number_events_per_transaction))
# fancy_histogram(fig2,ax2,number_meter_points_per_transaction,bins,linestyle="-", color=color_dic["siemens_colors_graphs"][0],weights=np.ones(len(number_meter_points_per_transaction)))
ax2.hist(number_meter_points_per_transaction, bins=bins, edgecolor=edgecolor)

diff = 1/3
bins = np.arange(0, np.max(duration_of_transactions)+diff/2, diff)
ax3.hist(duration_of_transactions, bins=bins, edgecolor=edgecolor)

# ax.set_u
props = dict(boxstyle='round', facecolor=color_dic["siemens_colors"][1], alpha=0.5)
for a in axs:
    a.set_ylabel("# Transactions", fontsize=fontsize)
    a.set_yscale('log')
    ymin, ymax = a.get_ylim()
    a.set_ylim(ymin,1e5)

ax.set_xlim(-0.5, 17.5)
ax.set_xlabel(r"# Logged events ($\cdot 10^3$)", fontsize=fontsize)
ax.set_title("Total number of logged events per transaction",fontsize=fontsize)

ax2.set_xlabel("# Meter Points", fontsize=fontsize)
ax2.set_title("Total number of meter points per transaction",fontsize=fontsize)

percent_of_zero_events = len([element for element in number_events_per_transaction if element == 0])/len(number_events_per_transaction)*100
percent_of_zero_meter = len([element for element in number_meter_points_per_transaction if element == 0])/len(number_meter_points_per_transaction)*100

percent_of_small_events = len([element for element in number_events_per_transaction if 0<=element < 0.05])/len(number_events_per_transaction)*100
small_val = 20
percent_of_small_meter = len([element for element in number_meter_points_per_transaction if element <= small_val])/len(number_meter_points_per_transaction)*100


print(percent_of_zero_meter, "% of transactions have no meter information" )
print(percent_of_zero_events, "% of transactions have no events information" )

print(percent_of_small_events, "% of transactions have less than 50 logged events" )
print(percent_of_small_meter, f"% of transactions have less than {small_val} meter points" )

ax3.xaxis.set_minor_locator(ticker.MultipleLocator(1/3))

ax3.set_xlim(-2,50)

if SAVE:
    fig.tight_layout()
    fig.savefig(os.path.join(SAVE_PATH_MANUAL, "NumberEventsHistogram.svg"))
    fig2.tight_layout()
    fig2.savefig(os.path.join(SAVE_PATH_MANUAL, "NumberMeterValuesHistogram.svg"))

if not SHOW:
    for a in axs:
        plot.close()

In [ ]:
# Zoomed number of events per transaction
fig, ax = plot.subplots()
fig2, ax2 = plot.subplots()

axs= [ax,ax2]
figs= [fig,fig2]
colors = color_dic["siemens_colors"]
prepare_subplots_big(axs,key="siemens_colors_graphs", background_color=colors[0], tick_color=colors[1], grid_color=colors[2], MINOR=True)
for f in figs:
    f.set_facecolor("none")

edgecolor = np.array(color_dic['siemens_colors_graphs'])[9]

SHOW = False
SAVE = False

number_events_per_transaction = [ len(transaction['events']) for name in name_of_chargers for transaction in all_transaction_info[name]  ]

number_meter_points_per_transaction = [ len(transaction['meter_values']) for name in name_of_chargers for transaction in all_transaction_info[name]  ]

diff = 1
bins = np.arange(0, 50+diff/2, diff)

# ax.hist(number_events_per_transaction,bins=bins)
# fancy_histogram(fig,ax,number_events_per_transaction,bins=bins, color=color_dic["siemens_colors_graphs"][0])
ax.hist(number_events_per_transaction, bins=bins, color=color_dic["siemens_colors_graphs"][0], weights=np.full(len(number_meter_points_per_transaction), 100)/len(number_meter_points_per_transaction), edgecolor=edgecolor)

diff = 1
bins = np.arange(0, 20, diff)
# fancy_histogram(fig2,ax2,number_meter_points_per_transaction,bins=bins, color=color_dic["siemens_colors_graphs"][0])
ax2.hist(number_meter_points_per_transaction, bins=bins, color=color_dic["siemens_colors_graphs"][0], weights=np.full(len(number_meter_points_per_transaction), 100)/len(number_meter_points_per_transaction), edgecolor=edgecolor)

for a in axs:
    a.set_ylabel("# Transactions", fontsize=fontsize)
    a.set_ylim(0,50)

ax.set_xlabel(r"# Logged events", fontsize=fontsize)
ax.set_title("Total number of logged events per transaction",fontsize=fontsize)
ax.set_xlim(-1,50)

ax2.set_xlabel("# Meter Points", fontsize=fontsize)
ax2.set_title("Total number of meter points per transaction",fontsize=fontsize)
ax2.set_xlim(-1,20)

ax2.xaxis.set_major_locator(ticker.MultipleLocator(5))

for f in figs:
    f.tight_layout()

fig.savefig(os.path.join(SAVE_PATH_MANUAL, "ZoomNumberEventsHistogram.svg"))
fig2.savefig(os.path.join(SAVE_PATH_MANUAL, "ZoomNumberMeterValuessHistogram.svg"))

if not SHOW:
    for a in axs:
        plot.close()
    

## Energy Delivered plots

In [ ]:
# Plot energy delivered %
# Depends on some variables above


SHOW = False
SAVE = False
fig, ax  = plot.subplots()
ax_twin = ax.twinx()
axs = [ax,ax_twin]
figs = [fig]
colors = color_dic["siemens_colors"]
prepare_subplots_big(axs,key="siemens_colors_graphs", background_color=colors[0], tick_color=colors[1], grid_color=colors[2])
for f in figs:
    f.set_facecolor("none")



diff_energy = 2.5
bins = np.arange(min_threshold_energy, max_threshold_energy, diff_energy)
hist = np.histogram(energy_delivered_for_filtered_transactions, bins=bins)

ys = hist[0]/len(energy_delivered_for_filtered_transactions)*100
ys_cumsum = np.cumsum(ys)


# ax.bar( hist[1][:-1], ys, width=diff_energy, align='edge', edgecolor="k")
fancy_histogram(fig, ax, energy_delivered_for_filtered_transactions, bins, color=color_dic['siemens_colors_graphs'][0], alpha=0.3)
ax_twin.step(np.concat([[0], hist[1][:-1]]), np.concat([[0], ys_cumsum]), color=color_dic['siemens_colors_graphs'][1])
ax_twin.set_ylim(0,100)
ax.set_ylim(0,10)
half_energy_index = np.argmin(np.abs(ys_cumsum-50))
ax_twin.plot([hist[1][half_energy_index],1e4],[ys_cumsum[half_energy_index]]*2, linestyle=":", color=color_dic['siemens_colors_graphs'][6], alpha=1)
ax_twin.plot([hist[1][half_energy_index]]*2,[0,ys_cumsum[half_energy_index]], linestyle=":", color=color_dic['siemens_colors_graphs'][6], alpha=1)


ax.set_xlim(-5,100)
# arrange_twin_plots(ax, ax_twin)
ax.set_xlabel(r"Energy Delivered (kWh)", fontsize=fontsize)
ax.set_ylabel(r"Transactions (%)", fontsize=fontsize)
ax_twin.set_ylabel(r"Cumulative Sum (%)", fontsize=fontsize)
ax.set_title("Filtered energy delivered distribution ",fontsize=fontsize)
print("Half of the energy is for: ", hist[1][half_energy_index])
print("Max energy: ", np.max(energy_delivered_for_filtered_transactions))
print(len([i for i in energy_delivered_for_filtered_transactions if i>100])/len(energy_delivered_for_filtered_transactions)*100)

# ax_twin.annotate( f"{hist[1][half_energy_index]}"+r"$\cdot 10^3$", (hist[1][half_energy_index],ys_cumsum[half_energy_index]), (hist[1][half_energy_index]-7.7,80), 
#             arrowprops=dict(arrowstyle="-", linestyle=":", color="r"), color="r", size=12)
ax.annotate(hist[1][half_energy_index], (hist[1][half_energy_index],0.1), (hist[1][half_energy_index], -0.52), va="center", ha="center", color=color_dic['siemens_colors_graphs'][6], transform=ax.get_transform(),size=fontsize, arrowprops={"arrowstyle":"-", "color":"r", "linewidth": 2.1})
ax_twin.annotate("{:.0f}".format(ys_cumsum[half_energy_index]), (100,ys_cumsum[half_energy_index]), (104.7,ys_cumsum[half_energy_index]), va="center", ha="center", color=color_dic['siemens_colors_graphs'][6], transform=ax_twin.get_transform(),size=fontsize, arrowprops={"arrowstyle":"-", "color":"r", "linewidth": 2.1})
for f in figs:
    f.tight_layout()
if SAVE:
    fig.subplots_adjust(left=0.1,right=0.88)
    fig.savefig(os.path.join(SAVE_PATH_MANUAL, "EnergyDeliveryDistribution.svg"))
if not SHOW:
    plot.close()



In [ ]:
# Energy Delivered vs Logged Events type Errror
fig, ax = plot.subplots()
fig2, ax2 = plot.subplots()
axs = [ax, ax2]

SHOW = False

figs = [fig, fig2]
colors = color_dic["siemens_colors"]
prepare_subplots(axs,key="siemens_colors_graphs", background_color=colors[0], tick_color=colors[1], grid_color=colors[2])
for f in figs:
    f.set_facecolor("none")

event_name = 'Error_Event'
energy_delivered_array = np.array( [element[0] for element in energy_delivered_info])
temp_num_logged_events = np.array( [element[2][event_name] for element in energy_delivered_info] )
mask = [element <= 100 for element in energy_delivered_array]
energy_delivered_array *= 1000

ax.scatter( temp_num_logged_events[mask], energy_delivered_array[mask], marker="o", alpha=0.8)
ax2.scatter( temp_num_logged_events, energy_delivered_array, marker="o", alpha=0.8)

for a in axs:
    a.set_title("Energy delivered vs number of events type \'Error\'", fontsize=fontsize)
    a.set_xlabel("Number of events logged", fontsize=fontsize)
    # a.set_ylabel(r"Energy delivered ($\cdot 10^3$)", fontsize=fontsize)
    a.set_ylabel(r"Energy delivered (kWh)", fontsize=fontsize)

color = color_dic['default'][0]
color = "k"
# ax.annotate("", (20,40), (10,80), 
#                 arrowprops=dict(arrowstyle="-|>", color=color, lw=2), color=color, size=15)

# ax2.annotate("", (15,200), (5,400), 
#                 arrowprops=dict(arrowstyle="-|>", color=color, lw=2), color=color, size=15)
ax.set_yscale('log')
ax2.set_yscale('log')

for a in axs:
    if not SHOW:
        plot.close()

In [ ]:
# Violin plots Energy delivered vs Logged Events type Error
fig, ax = plot.subplots()
fig2, ax2 = plot.subplots()
axs = [ax, ax2]
figs = [fig, fig2]
colors = color_dic["siemens_colors"]
prepare_subplots(axs,key="siemens_colors_graphs", background_color=colors[0], tick_color=colors[1], grid_color=colors[2])
for f in figs:
    f.set_facecolor("none")

SHOW = False
SAVE = False

event_name = 'Error_Event'
energy_delivered_array = np.array( [element[0] for element in energy_delivered_info]) 
temp_num_logged_events = np.array( [element[2][event_name] for element in energy_delivered_info] )

colors = color_dic['mega big2']

for i in range(20):
    mask = [element == i for element in temp_num_logged_events]
    if i <= 10:
        violin_parts = ax.violinplot(energy_delivered_array[mask],showextrema=False, positions=[i])
        for pc_index, pc in enumerate(violin_parts['bodies']):
            pc.set_facecolor(colors[i])
            pc.set_edgecolor("k")
        boxprop= {"linewidth": 1.5, "color": colors[i]}
        boxplot = ax2.boxplot(energy_delivered_array[mask], positions=[i], showfliers=False, widths=0.3, medianprops={"linewidth":0}, boxprops= boxprop, whiskerprops=boxprop, capprops=boxprop)
        ax.scatter(i, np.mean(energy_delivered_array[mask]), color=colors[i], edgecolors="k", linewidths=0.5,zorder=10)
        ax2.scatter(i, np.mean(energy_delivered_array[mask]), color=colors[i], edgecolors="k", linewidths=0.5,zorder=10)
    else:
        boxprop= {"linewidth": 0, "color": colors[i]}
        boxplot = ax2.boxplot(energy_delivered_array[mask], positions=[i], showfliers=False, widths=0.3, medianprops={"linewidth":0}, boxprops= boxprop, whiskerprops=boxprop, capprops=boxprop)
        ax.scatter([i]*len(energy_delivered_array[mask]), energy_delivered_array[mask], marker="s", color=colors[i], edgecolors="k", linewidths=0.5)
        ax2.scatter([i]*len(energy_delivered_array[mask]), energy_delivered_array[mask], marker="s", color=colors[i], edgecolors="k", linewidths=0.5)

markers = [
    mlines.Line2D([],[], linestyle="none", color="k", label="Mean", marker="o"),
    mlines.Line2D([],[], linestyle="none", color="k", label="Datapoint", marker="s"),
]

ax.xaxis.set_major_locator(ticker.MultipleLocator(5))
ax2.xaxis.set_major_locator(ticker.MultipleLocator(5))
ax2.set_xticklabels( [0, 0, 5, 10, 15])
for a in axs:
    a.set_title("Energy delivered vs number of events type \'Error\'", fontsize=fontsize)
    a.set_xlabel("Number of events logged", fontsize=fontsize)
    a.set_ylabel(r"Energy delivered (kWh)", fontsize=fontsize)

# ax.set_yscale('log')
ax.set_ylim(0, 1000)
ax.legend(handles=markers)

ax.set_ylim(1e-1, 1e3)
ax.set_yscale('log')

for f in figs:
    f.tight_layout()

if SAVE:
    fig.savefig(os.path.join(SAVE_PATH_MANUAL, "NumberErrorsVsEnergyDelivered.svg"))

if not SHOW:
    for a in axs:
        plot.close()

In [ ]:
# Histogram of Energy delivered vs logged events type error
fig, ax = plot.subplots()
axs = [ax]
figs = [fig]

SHOW = False
colors = color_dic["siemens_colors"]
prepare_subplots(axs, background_color=colors[0], tick_color=colors[1], grid_color=colors[2])
for f in figs:
    f.set_facecolor(color_dic["siemens_colors"][0])

event_name = 'Error_Event'
energy_delivered_array = np.array( [element[0] for element in energy_delivered_info]) * 1000
temp_num_logged_events = np.array( [element[2][event_name] for element in energy_delivered_info] )

colors = color_dic['default']

mask = np.array([element == 0 for element in temp_num_logged_events])

diff = 10000
bins = np.arange(0,np.max(energy_delivered_array)+diff/2, diff)
bins = np.logspace(0, np.log10(np.max(energy_delivered_array)), 100)

patches = []

for i in range(4):
    mask = np.array([5*i <= element < 5*(i+1) for element in temp_num_logged_events])
    fancy_histogram(fig, ax, energy_delivered_array[mask], bins, color=colors[i])
    patches.append(mpatches.Patch(facecolor=colors[i], label=f"{i*5:.0f}<= E < {(i+1)*5}"))


ax.set_xlim(1e3,1e6)
ax.set_xscale('log')
ax.legend(handles=patches, title="# Events logged")

ax.set_xlabel("Charging Rate", fontsize=fontsize)
ax.set_ylabel(" Tranasactions (%)", fontsize=fontsize)
ax.set_title("Distribution of charging rate per event of type \'Error\'")


if not SHOW:
    for a in axs:
        plot.close()

In [ ]:
# Difference between faulty and non_faulty energy deliver

SHOW = False
SAVE = False
fig, ax = plot.subplots()
axs = [ax]
figs = [fig]
colors = color_dic["siemens_colors"]
prepare_subplots_big(axs, MINOR=False, background_color=colors[0], tick_color=colors[1], grid_color=colors[2])
for f in figs:
    # f.set_facecolor(color_dic["siemens_colors"][0])
    f.set_facecolor("none")


faulty_chargers = ['znOp3r', 'hpcrTh', 'm5DaZZ', '1hPsjR', '8vobe9', 'mPNYK3', 'UlTc32', 'pI1V6h', '0AQbZI', 'NNNKEW', 'cbk8c9', 'IhfNTu', 'lpuy1j', 'VkLemc', '8wVfxj']

non_faulty = [name for name in name_of_chargers if not name in faulty_chargers]

energy_delivered_faulty = [ energy for name in faulty_chargers for energy in energy_delivered_per_charger[name] ]
energy_delivered_non_faulty = [ energy for name in non_faulty for energy in energy_delivered_per_charger[name] ]

new_energy_faulty = [energy for name in faulty_chargers for energy in energy_delivered_per_charger[name] if energy < 100]
print(np.mean(new_energy_faulty))

print(len(energy_delivered_faulty) - len(new_energy_faulty))

diff_energy = 5
bins = np.arange(0, 700, diff_energy)

# ax.hist(energy_delivered_non_faulty,bins=bins, weights=np.full(len(energy_delivered_non_faulty),100)/len(energy_delivered_non_faulty), histtype="step", color=color_dic['siemens_colors_graphs'][0], label="Non Faulty")

# ax.hist(energy_delivered_faulty,bins=bins, weights=np.full(len(energy_delivered_faulty),100)/len(energy_delivered_faulty), histtype="step", color=color_dic['siemens_colors_graphs'][6], label="Faulty")

fancy_histogram(fig,ax, energy_delivered_non_faulty,bins,color=color_dic['siemens_colors_graphs'][0], alpha=0.3)
fancy_histogram(fig,ax, energy_delivered_faulty,bins,color=color_dic['siemens_colors_graphs'][6], alpha=0.3)   

print("This is the mean of the normal case: ", np.mean(energy_delivered_non_faulty), r" $\pm$ ", np.std(energy_delivered_non_faulty)/np.sqrt(len(energy_delivered_non_faulty)))
print("This is the mean of the faulty case: ", np.mean(energy_delivered_faulty), r" $\pm$ ", np.std(energy_delivered_faulty)/np.sqrt(len(energy_delivered_faulty)))


props = dict(boxstyle='round', facecolor=color_dic["siemens_colors"][1], alpha=0.5)
patches = [
    mpatches.Patch(facecolor=color_dic['siemens_colors_graphs'][0], label="Normal", edgecolor="k"),
    mpatches.Patch(facecolor=color_dic['siemens_colors_graphs'][6], label="Faulty", edgecolor="k")
]

ax.set_xlim(-5,100)

ax.legend(handles=patches)
ax.text(0.71, 0.8, f"# Transactions: \nNormal: {len(energy_delivered_non_faulty)} \nFaulty: {len(energy_delivered_faulty)}", transform=ax.transAxes, fontsize=13, verticalalignment="top", bbox=props)

ax.set_xlabel(r"Energy Delivered (kWh)", fontsize=fontsize)
ax.set_ylabel(r"Transactions (%)", fontsize=fontsize)
ax.set_title("Energy delivered distribution for Faulty an Normal chargers")

for f in figs:
    f.tight_layout()

if SAVE:
    fig.savefig(os.path.join(SAVE_PATH_MANUAL, "EnergyDistributionNormalFaulty.svg"))
    fig.savefig(os.path.join(SAVE_PATH_MANUAL, "EnergyDistributionNormalFaulty.png"), dpi=620)
    pass

if not SHOW:
    for a in axs:
        plot.close()


## Probability

In [ ]:
# The most common events logged during the start of a charging period

SHOW = False
fig, ax = plot.subplots(figsize=(6,5))
fig2, ax2 = plot.subplots(figsize=(6,5))
axs = [ax,ax2]
prepare_subplots(axs)


    
diff = 5
bins = np.arange(0,5*60+5, diff)
show_only = 3

make_events_plot(fig, ax,
                 events_near_start,
                 events_near_start_statistic,
                 show_only, bins, diff,
                 total_metered_transactions)

ax.set_title("Time distribution of events with logged time close to \'Start charge\'", fontsize=fontsize)
ax.set_xlabel(" Difference between logged event and \'start charghing\'(seconds)", fontsize=fontsize)
ax.set_ylabel(" Logged Events (%) ", fontsize=fontsize)
ax.set_ylim(0,60)
ax.set_xlim(-10,150)

if not SHOW:
    plot.close()

diff = 10
bins = np.arange(0,5*60+5, diff)
show_only = 4
    
make_events_plot(fig2, ax2,
                 events_near_end,
                 events_near_end_statistic,
                 show_only, bins, diff,
                 total_metered_transactions, loc=(1,0.62))

ax2.set_title("Time distribution of events with logged time close to \'End charge\'", fontsize=fontsize)
ax2.set_xlabel(" Difference between logged event and \'end charghing\'(seconds)", fontsize=fontsize)
ax2.set_ylabel(" Logged Events (%) ", fontsize=fontsize)
ax2.set_ylim(0,50)
ax2.set_xlim(-10,310)


if not SHOW:
    plot.close()

In [ ]:
# Probability of transaction resulting in fault, when an event is logged
fig, ax = plot.subplots(constrained_layout=True,figsize=(12,6))
SHOW = False
SAVE = False
axs = [ax]
figs = [fig]

colors = color_dic["siemens_colors"]
prepare_subplots_big(axs, key="siemens_colors_graphs", background_color=colors[0], tick_color=colors[1], grid_color=colors[2])
for f in figs:
    f.set_facecolor("none")

edgecolor = np.array(color_dic['siemens_colors_graphs'])[9]
show_only = 15

list_of_array = [0,1,3,4,5,12,16,18,20]
keys = np.array(list(prob_fault_if_event.keys()))[list_of_array] 
ax.barh(keys, np.array(list(prob_fault_if_event.values()))[list_of_array], edgecolor=edgecolor)
for key_index, key in enumerate(np.array(list(prob_fault_if_event))[list_of_array]):
    ax.annotate(f"{np.sum([types_of_errors[i][key] for i in range(2)])} events logged", (prob_fault_if_event[key]+0.03, key_index -0.12), color=colors[1], fontsize=12)



ax.set_xticklabels(ax.get_xticklabels()[:6])


ax.set_xlabel("Probability", fontsize=18)
ax.set_title(" Events that likely result in \'EndResult\'== Fault", fontsize=18)
ax.set_xlim(0,1.4)

if SAVE:
    fig.savefig(os.path.join(SAVE_PATH_MANUAL, "ProbabilityFaultGivenError.svg"))

if not SHOW:
    plot.close()

In [ ]:
# Make a plot with the probability of charger make a fault

fig, ax = plot.subplots()
fig2, ax2 = plot.subplots()
fig3, ax3 = plot.subplots()
fig4, ax4 = plot.subplots()
fig5, ax5 = plot.subplots()
axs = [ax, ax2, ax3, ax4, ax5]
figs = [fig, fig2, fig3, fig4, fig5]
colors = color_dic["siemens_colors"]
prepare_subplots_big(axs, MINOR=False, background_color=colors[0], tick_color=colors[1], grid_color=colors[2])
for f in figs:
    # f.set_facecolor(color_dic["siemens_colors"][0])
    f.set_facecolor("none")

SHOW = True
SAVE = False
probs_of_fault = np.array([
     types_of_transactions[name][0]/(1e-6+np.sum(types_of_transactions[name]))
    for name in name_of_chargers
])

print("These are the chargers faulty: ")
print([name for name, prob in zip(name_of_chargers, probs_of_fault) if prob <0.8])

# probs_of_fault = np.array([
#      durations_of_transactions[name][0]/(1e-6+np.sum(durations_of_transactions[name]))
#     for name in name_of_chargers
# ])
    
total_number_transactions = np.array([
    np.sum(types_of_transactions[name])
    for name in name_of_chargers
])

diff =0.05
lim = 0.8
colors = np.array(color_dic['default'])[[0,4]]
colors = np.array(color_dic['siemens_colors_graphs'])[[0,6]]
edgecolor = np.array(color_dic['siemens_colors_graphs'])[8]
edgecolor = np.array(color_dic['siemens_colors_graphs'])[9]

bins = np.arange(0,1.01,diff)
height = np.zeros(len(bins))
hist = np.histogram(probs_of_fault, bins=bins)

mask = np.array([element >= lim for element in hist[1][:-1] ])
ax.bar(hist[1][:-1][mask], hist[0][mask], color=colors[0], alpha=1, width=diff, align="edge",edgecolor=edgecolor)
ax.bar(hist[1][:-1][~mask], hist[0][~mask], color=colors[1], alpha=1, width=diff, align="edge",edgecolor=edgecolor)

most_common_errors = []
faulty_chargers_index = []
for index_name, name in enumerate(name_of_chargers):
    if probs_of_fault[index_name] < 0.8:
        faulty_chargers_index.append(index_name)
        for key in unhealthy_chargers[name]:
            most_common_errors.append(key[0])
most_common_errors = sorted(Counter(most_common_errors).items(), key= lambda x: x[1], reverse=True)

print(most_common_errors)

event_names = ["Error_Event", "outletX", "Outlet_DC_codes (X)", "UnknownError"]

if True:
    data_normal = [
        [freq_errors_per_charger[event_name][i] for i in range(len(name_of_chargers)) if not (i in faulty_chargers_index)]
        for event_name in event_names
    ]

    data_faulty = [
        [freq_errors_per_charger[event_name][i] for i in faulty_chargers_index]
        for event_name in event_names
    ]

    space = 0.03
    width = 0.35

    ax2.bar(event_names, np.mean(data_normal, axis=1), width=-width, align="edge", color=colors[0], edgecolor=edgecolor)
    ax2.errorbar([i-width/2 for i in range(len(event_names))], np.mean(data_normal, axis=1), yerr=np.std(data_normal, axis=1)/np.sqrt((len(name_of_chargers)-len(faulty_chargers_index))),color=edgecolor, linestyle="none", marker=".", markerfacecolor=colors[0])

    ax2.bar(event_names, np.mean(data_faulty, axis=1), width=width, align="edge", color=colors[1], edgecolor=edgecolor)
    ax2.errorbar([i+width/2 for i in range(len(event_names))], np.mean(data_faulty, axis=1), yerr=np.std(data_faulty, axis=1)/np.sqrt((len(faulty_chargers_index))),color=edgecolor, linestyle="none", marker=".", markerfacecolor=colors[1])

    violin_parts = ax3.violinplot(data_normal, positions=[i-space for i in range(len(data_normal))], showextrema=False, showmedians=False)

    for pc_index, pc in enumerate(violin_parts['bodies']):
        # get the center
        m = np.mean(pc.get_paths()[0].vertices[:, 0])
        # modify the paths to not go further right than the center
        pc.get_paths()[0].vertices[:, 0] = np.clip(pc.get_paths()[0].vertices[:, 0], -np.inf, m)
        pc.set_facecolor(colors[0])
        pc.set_edgecolor(colors[0])

    violin_parts = ax3.violinplot(data_faulty, positions=[i+space for i in range(len(data_normal))], showextrema=False, showmedians=False)

    for pc_index, pc in enumerate(violin_parts['bodies']):
        # get the center
        m = np.mean(pc.get_paths()[0].vertices[:, 0])
        # modify the paths to not go further right than the center
        pc.get_paths()[0].vertices[:, 0] = np.clip(pc.get_paths()[0].vertices[:, 0], m, np.inf)
        pc.set_facecolor(colors[1])
        pc.set_edgecolor(colors[1])

    for i in range(len(data_normal)):
        ax3.scatter([i-space for _ in range(len(data_normal[i]))], data_normal[i], color=colors[0], alpha=0.3, marker=".")
        ax3.scatter([i+space for _ in range(len(data_faulty[i]))], data_faulty[i], color=colors[1], alpha=0.3, marker=".")


    ax3.errorbar([i-space*5 for i in range(len(event_names))], np.mean(data_normal, axis=1), yerr=np.std(data_normal, axis=1)/np.sqrt((len(name_of_chargers)-len(faulty_chargers_index))),color=edgecolor, linestyle="none", marker="<", markerfacecolor=colors[0], markeredgewidth=0.5, linewidth=1.5)
    ax3.errorbar([i+space*5 for i in range(len(event_names))], np.mean(data_faulty, axis=1), yerr=np.std(data_faulty, axis=1)/np.sqrt((len(faulty_chargers_index))),color=edgecolor, linestyle="none", marker=">", markerfacecolor=colors[1], markeredgewidth=0.5,linewidth=1.5)

    mask = np.array([i in faulty_chargers_index for i in range(len(name_of_chargers))])
    event_name = 'Error_Event'
    ys = np.array(freq_errors_per_charger[event_name])
    ax4.scatter(probs_of_fault[~mask], ys[~mask], color=colors[0], marker="o")
    ax4.scatter(probs_of_fault[mask], ys[mask], color=colors[1], marker="o")

    ax5.scatter(total_number_transactions[~mask], ys[~mask], color=colors[0], marker="o")
    ax5.scatter(total_number_transactions[mask], ys[mask], color=colors[1], marker="o")
    fancy_plot(ax5, total_number_transactions, ys, event_name)

    ax5.set_title(f"Event: \'{event_name}\'",size=fontsize)
    ax5.set_xlabel("# Transactions logged", size=fontsize)
    ax5.set_ylabel("Event Frequency", size=fontsize)

    patches = [
        mpatches.Patch(facecolor=colors[0], label="Normal", edgecolor="k"),
        mpatches.Patch(facecolor=colors[1], label="Faulty", edgecolor="k")
    ]

    markers = [
        mlines.Line2D([],[], marker="^", color="k", label="mean"),
        mlines.Line2D([],[], marker=".", color="k", label="datapoint", linestyle="none")
    ]

    ax.set_xlabel(r"$\frac{\text{Successful Transactions}}{\text{Total Transactions}}$", fontsize=fontsize+4)
    ax.set_ylabel(r"Number of chargers", fontsize=fontsize)
    ax.set_title(r"Distribution of transaction status", fontsize=fontsize)

    ax2.set_ylabel("Event Frequency", fontsize=fontsize)
    ax2.tick_params(axis="x", rotation=0)

    ax3.xaxis.set_major_locator(ticker.MultipleLocator(1))
    ax3.set_xticklabels([""] + [event for event in event_names])
    ax3.set_ylabel("Event Frequency", fontsize=fontsize)
    ax3.tick_params(axis="x", rotation=15)
    ax3.set_title(r"Event Frequency distribution", fontsize=fontsize)

    ax4.set_title("Frequency of events of type \'Error\' vs relative fault transactions", fontsize=fontsize)
    ax4.set_ylabel("Event Frequency", fontsize=fontsize)
    ax4.set_xlabel(r"$\frac{\text{Successful Transactions}}{Total Transactions}$", fontsize=fontsize)

    for a in axs[1:-1]:
        leg = a.legend(handles=patches)
        a.add_artist(leg)
        a.legend(handles=markers, bbox_to_anchor=(1.0,0.83), bbox_transform=a.transAxes)

patches = [
    mpatches.Patch(facecolor=color_dic['siemens_colors_graphs'][0], label="Normal", edgecolor="k"),
    mpatches.Patch(facecolor=color_dic['siemens_colors_graphs'][6], label="Faulty", edgecolor="k")
]
ax.legend(handles=patches, fontsize=14)


for a in axs:
    a.title.set_fontname("Arial")
    a.xaxis.label.set_fontname("Arial")
    a.yaxis.label.set_fontname("Arial")
    for lbl in a.get_xticklabels() + a.get_yticklabels():
        lbl.set_fontname("Arial")
    leg = a.get_legend()
    if leg:
        for text in leg.get_texts():
            text.set_fontname("Arial")

# ax4.set_ylim(-0.01,0.5)
if not SHOW:
    for a in axs:
        plot.close()

for f in figs:
    f.tight_layout()
if SAVE:
    fig.savefig(os.path.join(SAVE_PATH_MANUAL, "HistogramFaultyChargers.svg"))

    fig3.savefig(os.path.join(SAVE_PATH_MANUAL, "ViolinPlotsEventFrequencyVsFaulty.svg"))


## Charging Rate

In [ ]:
# Energy Delivered vs Charging time
fig, ax = plot.subplots()
axs = [ax]
prepare_subplots(axs)
SHOW = False

energy_delivered_array = [element[0] for element in energy_delivered_per_charging_times]
charging_time = [element[1] for element in energy_delivered_per_charging_times]



ax.scatter(energy_delivered_array, charging_time)

ax.set_xlabel(r" Energy Delivered ($\cdot 10^3$)", fontsize=fontsize)
ax.set_ylabel("Charging Time (hours)", fontsize=fontsize)




# ax.set_ylim(-0.1, 100)
# ax.set_xlim(-5, 100)


if not SHOW:
    plot.close()


In [ ]:
# Charging Rate vs Number of logged events
fig, ax = plot.subplots()
fig2, ax2 = plot.subplots()
axs = [ax, ax2]
figs = [fig, fig2]
colors = color_dic["siemens_colors"]
prepare_subplots(axs,key="siemens_colors_graphs", background_color=colors[0], tick_color=colors[1], grid_color=colors[2])
for f in figs:
    f.set_facecolor("none")

SHOW = 0
SAVE = False

charging_rate_array = np.array([element[0]/(element[1]+1e-5) for element in energy_delivered_per_charging_times])
event_name = 'Error_Event'
temp_num_logged_events = np.array( [element[3][event_name] for element in energy_delivered_per_charging_times] )
bins = np.arange(0,100,1)

ax.scatter(temp_num_logged_events, charging_rate_array)

colors = color_dic["mega big2"]
# colors = color_dic["scg"]
for i in range(20):
    mask = [element == i for element in temp_num_logged_events]
    if i <= 10:
        violin_parts = ax2.violinplot(charging_rate_array[mask], positions=[i], showextrema=False)
        for pc_index, pc in enumerate(violin_parts['bodies']):
            pc.set_facecolor(colors[i])
            pc.set_edgecolor("k")
        ax2.scatter(i, np.mean(charging_rate_array[mask]),color=colors[i])
    else:
        ax2.scatter([i]*len(charging_rate_array[mask]), charging_rate_array[mask], marker="s", color=colors[i])


for a in axs:
    a.set_title("Rate of charge vs number of events type \'Error\'", fontsize=fontsize)
    a.set_xlabel("Number of events logged", fontsize=fontsize)
    a.set_ylabel("Rate of charge (kW)", fontsize=fontsize)
    a.set_yscale("log")

markers = [
    mlines.Line2D([],[], linestyle="none", color="k", label="Mean", marker="o"),
    mlines.Line2D([],[], linestyle="none", color="k", label="Datapoint", marker="s"),
]

# ax2.set_ylim(1e4, 2e5)
ax2.xaxis.set_major_locator(ticker.MultipleLocator(5))
ax2.legend(handles=markers)

for f in figs:
    f.tight_layout()

if SAVE:
    fig2.savefig(os.path.join(SAVE_PATH_MANUAL, "NumberOfErrorsvsRateOfCharge.svg"))

if not SHOW:
    for a in axs:
        plot.close()

In [ ]:
# Distribution of charging rate
fig, ax = plot.subplots()
# ax_twin = ax.twinx()
fig2, ax2 = plot.subplots()
fig3, ax3 = plot.subplots()
axs = [ax,ax2,ax3]
figs = [fig, fig2, fig3]
SHOW = False
SAVE = False
colors = color_dic["siemens_colors"]
prepare_subplots(axs, key="siemens_colors_graphs", background_color=colors[0], tick_color=colors[1], grid_color=colors[2])
for f in figs:
    f.set_facecolor("none")

ax2.grid(True, color=colors[2], linestyle=((0,(2,5))), which="minor")

colors = color_dic["siemens_colors_graphs"]

charging_rate_array = np.array([element[0]/(element[1]+1e-5) for element in energy_delivered_per_charging_times])
charging_duration_array = np.array([element[1] for element in energy_delivered_per_charging_times])
charge_time_vs_transaction_time = np.array([element[4]/(element[1]+1e-5) for element in energy_delivered_per_charging_times])
mask = np.array([element[2] == "EVDisconnected" for element in energy_delivered_per_charging_times])

diff=5
bins = np.arange(0,250+diff/2,diff)
# ax.hist(charging_rate_array, bins=bins)

hist = np.histogram(charging_rate_array, bins=bins)
ys = hist[0]/len(charging_rate_array)*100
ys_cumsum = np.cumsum(ys)

fancy_histogram(fig, ax, charging_rate_array, bins=bins, color=colors[0],alpha=0.3)

smallest = 250
print( len([element for element in charging_rate_array if element< smallest])/(len(charging_rate_array)), f"of transactions have charging rate less than {smallest}")
# ax_twin.step(np.concat([[0], hist[1][:-1]]), np.concat([[0], ys_cumsum]), color=color_dic['siemens_colors_graphs'][1])
# ax_twin.set_ylim(0,100)
print("The mean charging rate is: ", np.mean(charging_rate_array), r"$\pm$", np.std(charging_rate_array)/len(charging_rate_array))

ax.set_xlabel(r"Rate of charge (kW)", fontsize=fontsize)
ax.set_ylabel("Transactions (%)", fontsize=fontsize)
ax.set_title("Rate of charge distribution", fontsize=fontsize)

fancy_histogram(fig, ax3, charging_rate_array[~mask], bins, color=colors[6],alpha=0.3)
fancy_histogram(fig, ax3, charging_rate_array[mask], bins, color=colors[0],alpha=0.3)
ax3.set_xlabel(r"Rate of charge (kW)", fontsize=fontsize)
ax3.set_ylabel("Transactions (%)", fontsize=fontsize)
ax3.set_title("Rate of charge distribution", fontsize=fontsize)


diff = 1/6
bins = np.arange(0,5+diff/2,diff)
fancy_histogram(fig2, ax2, charging_duration_array, bins, color=colors[0],alpha=0.3)
ax2.xaxis.set_minor_locator(ticker.MultipleLocator(1/6))
ax2.xaxis.set_major_locator(ticker.MultipleLocator(1))

ax2.set_xlabel("Charge Duration (min)", fontsize=fontsize)
ax2.set_ylabel("Transactions (%)", fontsize=fontsize)
ax2.set_title("Charge duration distribution", fontsize=fontsize)

ax2.yaxis.minorticks_off()

ax2.set_xlim(-0.25,5)
ax2.set_ylim(0.,25)


# ax.set_xlim(-10,250)
ax.set_ylim(0,8)


for f in figs:
    f.tight_layout()

if SAVE:
    fig.savefig(os.path.join(SAVE_PATH_MANUAL,"ChargingRateDistribution.svg"))
    fig2.savefig(os.path.join(SAVE_PATH_MANUAL,"DurationDistribution.svg"))


if not SHOW:
    for a in axs:
        plot.close()

In [ ]:
# Difference between transaction and charging durations.

charging_duration = np.array([element[1] for element in energy_delivered_per_charging_times])
transaction_duration = np.array([element[4] for element in energy_delivered_per_charging_times])
efficiency_metric = np.array([x/(y+1e-6) for x,y in zip(charging_duration, transaction_duration)])
SHOW = False
SAVE = False
fig, ax = plot.subplots()
fig2, ax2 = plot.subplots()
fig3, ax3 = plot.subplots()
axs  = [ax, ax2, ax3]
figs = [fig, fig2, fig3]
colors = color_dic["siemens_colors"]
prepare_subplots_big(axs,key="siemens_colors_graphs", background_color=colors[0], tick_color=colors[1], grid_color=colors[2])
for f in figs:
    f.set_facecolor("none")



smallest = 5
print( len([element for element in transaction_duration if element< smallest])/(len(transaction_duration)), f"of transactions have transaction_duration less than {smallest}")

print("The mean charge duration is: ", np.mean(charging_duration)*60, r"$\pm$", np.std(charging_duration)*60/len(charging_duration))

x_max = np.max(transaction_duration)+5

ax.scatter(transaction_duration, charging_duration, marker=".", zorder=3)#, label="Datapoint")
ax.fill_between([0, x_max], [0,0], [0,x_max], alpha=0.2, )
ax.plot([0,x_max],[0,x_max], linestyle="--", color=color_dic["siemens_colors_graphs"][1],linewidth=1)#,label="Efficient charging")
ax.plot([0,x_max],[0,0], linestyle="--", color=color_dic["siemens_colors_graphs"][6],linewidth=1)#,label="No charging")

diff = 1/6
bins = np.arange(0, 5, diff)
# ax2.hist(transaction_duration, bins=bins)
fancy_histogram(fig2, ax2, charging_duration, bins=bins, color=color_dic["siemens_colors_graphs"][0],alpha=0.3)

print("charging dur ", np.mean(charging_duration))
print("charging dur ", np.mean(charging_duration)*60)

ax.set_xlabel(" Transaction Duration (hrs)", fontsize=fontsize)
ax.set_ylabel(" Charging Duration (hrs)", fontsize=fontsize)
ax.set_title("Transaction vs charge duration", fontsize=fontsize)


min_, max_ = -0.2, 5
ax.set_xlim(min_,max_)
ax.set_ylim(min_,max_)

ax2.set_xlabel(" Charging Duration (hrs)", fontsize=fontsize)
ax2.set_ylabel(" Transactions (%)", fontsize=fontsize)
ax2.set_title(" Charging Duration distribution ", fontsize=fontsize)
ax2.set_ylim(0,25)

ax2.xaxis.set_major_locator(ticker.MultipleLocator(1))
ax2.xaxis.set_minor_locator(ticker.MultipleLocator(1/6))

diff = 0.02
bins = np.arange(0, 2+diff/2, diff)
# ax3.hist(efficiency_metric, bins=bins)

mask = [1-diff>element or element > 1+diff for element in efficiency_metric]

fancy_histogram(fig3, ax3, efficiency_metric[mask], bins=bins, color=color_dic["siemens_colors_graphs"][0], weights=np.full(len(efficiency_metric[mask]),100)/len(efficiency_metric), alpha=0.3)
ax3.hist(efficiency_metric[mask], bins=bins, color=color_dic["siemens_colors_graphs"][0], weights=np.full(len(efficiency_metric[mask]),100)/len(efficiency_metric), alpha=0.05,edgecolor=color_dic["siemens_colors_graphs"][0],zorder=1)
# ax3.set_yscale('log')
ax3.set_ylim(0,1)
ax3.set_xlim(-0.1,1.1)
print(len([i for i in efficiency_metric if i>0.95]), len(efficiency_metric))
print(len([i for i in efficiency_metric if 1.02>=i>0.98])/len(efficiency_metric))

ax3.fill_between([1-diff,1+diff], 0,5, alpha=0.3, hatch="/", color=color_dic['siemens_colors_graphs'][6])

ax3.set_xlabel(r" $\frac{\text{Charging Duration}}{\text{Transaction Duration}}$", fontsize=18)
ax3.set_ylabel(" Transactions (%)", fontsize=fontsize)
ax3.set_title(" Charging Time Efficiency", fontsize=fontsize)

# plot.scatter([i for i in range(len(efficiency_metric))], efficiency_metric)

for f in figs:
    f.tight_layout()
# fig.subplots_adjust(right=0.86)
if SAVE: 
    fig.savefig(os.path.join(SAVE_PATH_MANUAL, "TransactionvsChargeDuration.svg"))
    fig2.savefig(os.path.join(SAVE_PATH_MANUAL, "ChargingDurationDistribution.svg"))
    fig3.savefig(os.path.join(SAVE_PATH_MANUAL, "ChargingDurationVsTransactionDistribution.png"), dpi=620)

if not SHOW:
    for a in axs:
        plot.close()


In [ ]:
# Difference between faulty and non_faulty rate of charge

SHOW = 1
SAVE = False
fig, ax = plot.subplots()
axs = [ax]
figs = [fig]
colors = color_dic["siemens_colors"]
prepare_subplots_big(axs, MINOR=False, background_color=colors[0], tick_color=colors[1], grid_color=colors[2])
for f in figs:
    # f.set_facecolor(color_dic["siemens_colors"][0])
    f.set_facecolor("none")


faulty_chargers = ['znOp3r', 'hpcrTh', 'm5DaZZ', '1hPsjR', '8vobe9', 'mPNYK3', 'UlTc32', 'pI1V6h', '0AQbZI', 'NNNKEW', 'cbk8c9', 'IhfNTu', 'lpuy1j', 'VkLemc', '8wVfxj']

non_faulty = [name for name in name_of_chargers if not name in faulty_chargers]

rate_of_charge_faulty = [ energy/duration for name in faulty_chargers for energy,duration in zip(energy_delivered_per_charger[name], charging_duration_per_charger[name]) ]

rate_of_charge_non_faulty = [ energy/duration for name in non_faulty for energy,duration in zip(energy_delivered_per_charger[name], charging_duration_per_charger[name]) ]

diff = 10
bins = np.arange(0,250+diff/2,diff)

fancy_histogram(fig,ax, rate_of_charge_non_faulty,bins,color=color_dic['siemens_colors_graphs'][0], alpha=0.3)
fancy_histogram(fig,ax, rate_of_charge_faulty,bins,color=color_dic['siemens_colors_graphs'][6], alpha=0.3)   

props = dict(boxstyle='round', facecolor=color_dic["siemens_colors"][1], alpha=0.5)
patches = [
    mpatches.Patch(facecolor=color_dic['siemens_colors_graphs'][0], label="Normal", edgecolor="k"),
    mpatches.Patch(facecolor=color_dic['siemens_colors_graphs'][6], label="Faulty", edgecolor="k")
]

print("For faulty case: ", np.mean(rate_of_charge_faulty), r" $\pm$ ", np.std(rate_of_charge_faulty)/np.sqrt(len(rate_of_charge_faulty)))
print("For Normal case: ", np.mean(rate_of_charge_non_faulty), r" $\pm$ ", np.std(rate_of_charge_non_faulty)/np.sqrt(len(rate_of_charge_non_faulty)))

# ax.set_xlim(-5,100)

ax.legend(handles=patches, fontsize=14)
ax.text(0.72, 0.76, f"# Transactions: \nNormal: {len(rate_of_charge_non_faulty)} \nFaulty: {len(rate_of_charge_faulty)}", transform=ax.transAxes, fontsize=13, verticalalignment="top", bbox=props)

ax.set_xlabel(r"Rate of charge (kW)", fontsize=fontsize)
ax.set_ylabel(r"Transactions (%)", fontsize=fontsize)
ax.set_title("Rate of Charge distribution for Faulty an Normal chargers", fontsize=fontsize, pad=20)

ax.set_ylim(0,15)
ax.set_xlim(-10,260)

ax.yaxis.set_major_locator(ticker.MultipleLocator(5))

for a in axs:
    a.title.set_fontname("Arial")
    a.xaxis.label.set_fontname("Arial")
    a.yaxis.label.set_fontname("Arial")
    for lbl in a.get_xticklabels() + a.get_yticklabels():
        lbl.set_fontname("Arial")
    leg = a.get_legend()
    if leg:
        for text in leg.get_texts():
            text.set_fontname("Arial")

for f in figs:
    f.tight_layout()

if SAVE:
    fig.savefig(os.path.join(SAVE_PATH_MANUAL, "RateOfChargeNormalFaulty.svg"))
    fig.savefig(os.path.join(SAVE_PATH_MANUAL, "RateOfChargeNormalFaulty.png"), dpi=620)
    pass

if not SHOW:
    for a in axs:
        plot.close()


In [ ]:
# Correlations between charging rate vs number of logged events
print(list(charging_rate_correlations)[:4])
SHOW = False
fig, ax = plot.subplots()
fig2, ax2 = plot.subplots()
axs = [ax,ax2]
X = np.array([[element[3][key] for key in set(combine_error_dic.values())] for element in energy_delivered_per_charging_times])
y = np.array([element[0]/(1e-6+element[1]) for element in energy_delivered_per_charging_times])

event_index = list(set(combine_error_dic.values())).index("Error_Event")
event_index = 71
ax.scatter(X[:,event_index], y, label="{:.4f}".format(charging_rate_correlations[event_index]))
ax.set_title(f"{list(set(combine_error_dic.values()))[event_index]}")

for i in range(np.max(X[:,event_index])):
    mask = [x[event_index]== i for x in X]
    y_mask = y[mask]
    if len(y_mask):
        ax2.violinplot(y_mask, positions=[i])

for a in axs:
    a.set_xlabel("# Logged Events", fontsize=fontsize)
    a.set_ylabel("Charging Rate", fontsize=fontsize)
ax.legend(title="Correlation")

if not SHOW:
    for a in axs:
        plot.close()

## Chargers with Gaps

In [ ]:
fig, ax = get_subplots()
ax.step(steps[:,0], steps[:,1])
ax.tick_params(axis="x", rotation=90)
ax.set_ylabel("# chargers", fontsize=18)
ax.set_title("Number of chargers currently in data gap", fontsize=18)

# ax.set_xlim(pd.Timestamp("2025-12-21"),pd.Timestamp("2026-01-15"))
plot.show()

In [ ]:
fig, ax = get_subplots()
ax.hist(frac_in_disconnect,bins=np.arange(0,0.25,0.01))
ax.set_ylabel("Number of chargers")
ax.set_xlabel("")
plot.show()



In [ ]:
fig, ax = get_subplots(1,2, figsize=(15,5))

ax[0].hist(distance_to_min, bins=np.arange(-100, 101, 0.5))
ax[1].hist(distance_to_max, bins=np.arange(-100, 101, 0.5))

for a in ax:
    a.set_xlim(-10,10)

plot.show()


## Other

In [ ]:
# Events outside transactions
fig, ax = plot.subplots()
fig2, ax2 = plot.subplots()
axs = [ax, ax2]
prepare_subplots(axs)
SHOW = False

keys = set(types_of_errors[0].keys()).union( types_of_errors[1].keys() )
events_inside_transactions = {name: {key: (error_dic[0][key] + error_dic[1][key]) for key in keys} for name, error_dic in errors_per_charger.items()}

errors_outside = [events_outside_transactions[name]["Error_Event"] for name in name_of_chargers]
errors_inside = [events_inside_transactions[name]["Error_Event"] for name in name_of_chargers]

frac = [(i, errors_outside[i]/(errors_inside[i]+1e-1)) for i in range(len(errors_outside))]
frac_of_outside = [errors_outside[i]/(1e-5+errors_outside[i]+errors_inside[i]) for i in range(len(errors_outside))]

total_number_transactions = [np.sum(types_of_transactions[name]) for name in name_of_chargers]

indexes = (np.array([element > 0.9 for element in frac_of_outside]))
print(np.array(name_of_chargers)[indexes])
frac = sorted(frac, key=lambda x: x[1], reverse=True)

diff = 0.2
width = 0.35
bins = np.arange(0,10,diff)
show_only = 10
start = 0
# for index_name, name in enumerate(name_of_chargers):
colors = color_dic['default']
for index, _ in frac[start:start+show_only]:
        ax.barh(name_of_chargers[index], errors_outside[index], height=-width, align='edge', color=colors[0])
        ax.barh(name_of_chargers[index], errors_inside[index], height=width, align='edge', color=colors[1])
ax.set_xscale("log")

ax2.scatter(total_number_transactions, frac_of_outside)
ax2.set_xlabel("# Transactions", fontsize=fontsize)
ax2.set_ylabel(r"$\frac{\text{# Errors Outside Transactions}}{\text{# Total Errors}}$", fontsize=17)


if not SHOW:
    for a in axs:
        plot.close()

In [ ]:
# Connect Events to transactions
specific_name, specific_transaction_index = 'VaBjRf', 256
# Using the old method
fig, ax = plot.subplots()
SHOW = False
SAVE = False
axs = [ax]
figs = [fig]
colors = color_dic["siemens_colors"]
prepare_subplots_big(axs, MINOR=True, background_color=colors[0], tick_color=colors[1], grid_color=colors[2])
for f in figs:
    # f.set_facecolor(color_dic["siemens_colors"][0])
    f.set_facecolor("none")
grid_color = color_dic['siemens_colors'][1]

colors = color_dic['siemens_colors_graphs']

linestyles = ["-", "--"]
heights = [[0.0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0,0.5,0.47,0.47,0.0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8, 0.9,1.2,1.3,1.4],
            [0.0,0.1,0.2,0.53,0.3,0.4,0.6,0.7,0.8,0.9,0.7,0.1,0.2,0.3,0.4,0.6,0.8,0.53,1.8,1.7,1.7,1.9,1.1]]

for method_index, method in enumerate([all_transaction_info]):
    specific_transaction= method[specific_name][specific_transaction_index]
    specific_transaction_next= method[specific_name][specific_transaction_index+1]

    transactions = [specific_transaction, specific_transaction_next]
    for transaction_index in range(2):
        start = transactions[transaction_index]['transaction']['StartAt']
        end = transactions[transaction_index]['transaction']['EndAt']

        # lines = ax.plot([start, end], [transaction_index]*2, marker="o")
        ax.fill_between([start,end], [0,0], [1]*2, alpha=0.5, color=colors[transaction_index])
        transaction_events = transactions[transaction_index]['events']
        for event_index, event in enumerate(transaction_events):
            # ax.axvline(event['event_time'], color=lines[0].get_color(), linestyle=linestyles[transaction_index])
            if event['EvseID'] is None:
                ax.scatter(event['event_time'], heights[transaction_index][event_index], marker="x", facecolor=colors[transaction_index],zorder=4, edgecolor=color_dic['siemens_colors'][0], linewidths=2)
            else:
                ax.scatter(event['event_time'], heights[transaction_index][event_index], marker="|", color=colors[transaction_index],s=70,zorder=4, edgecolor=color_dic['siemens_colors'][0], linewidth=2)

marker = [
    mlines.Line2D([],[],marker="|",color="k", label="Assigned Events", linestyle="None"),
    mlines.Line2D([],[],marker="x",color="k", label="Ambiguous Events", linestyle="None"),
]




ax.axes.get_yaxis().set_visible(False)

ax.grid(True, color=grid_color,linestyle="--", linewidth=0.5,alpha=0.7, which='major', axis="x")
ax.grid(True, color=grid_color,linestyle=":", linewidth=0.5,alpha=0.5, which='minor', axis="x")

for a in axs:
    a.tick_params(axis='x', rotation=45)
    a.set_xticklabels([tick.get_text()[3:] for tick in a.get_xticklabels()])

start_check = pd.Timestamp("2025-12-01 10:47:00.000000+05:30")
end_check = pd.Timestamp("2025-12-01 10:58:00.000000+05:30")

print(all_transaction_info[specific_name][specific_transaction_index]['transaction'])
print(all_transaction_info[specific_name][specific_transaction_index]['events'][0])
for event in all_transaction_info[specific_name][specific_transaction_index]['events']:
    if start_check < event['event_time'] < end_check:
        print(event)
        print(" ")

print(all_transaction_info[specific_name][specific_transaction_index+1]['transaction'])
for event in all_transaction_info[specific_name][specific_transaction_index+1]['events']:

    if start_check < event['event_time'] < end_check and not event['EvseID'] == 1:
        print(event)
        print(" ")

ax.yaxis.set_major_locator(ticker.MultipleLocator(0.2))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(0.1))
ax.set_ylim(-0.05,1.25)
ax.legend(handles=marker, loc="upper right")
ax.set_title(f"Example of two concurrent transactions for charger {specific_name}", fontsize=fontsize)
ax.set_xlabel("Time of transaction", fontsize=fontsize)

for f in figs:
    f.tight_layout()

if SAVE:
    fig.savefig(os.path.join(SAVE_PATH_MANUAL, "ExampleOfSimultaneousTransaction.svg"))

if not SHOW:
    for a in axs:
        plot.close()


# Investigation

In [ ]:
some = [
    "910dd5ef-7ddb-419c-a73f-c386d2d249d9",
    "c7e4a0ee-9398-4fb0-9e52-04b038dc76fa",
    "ab70b25d-b3c7-4d8a-ad25-40df3d9dfce3",
    "9484454d-f6e9-4d4b-9654-377851fb4339",
    "05fddfc8-a321-4984-bb37-e3890ccb751e",
]

In [ ]:
transactions[[ True if np.any([i == element for element in some]) else False  for i in transactions['ChargerTransactionID'] ]][['ChargerID', "StartAt", "EndAt", "EndResult", "InitialSoC", "EndSoC", "EnergyDelivered", "ExtraInfo", "ChargerTransactionID"]]

In [ ]:
for index, i in enumerate(all_transaction_info['z3CHWb']):
    if i['transaction']["ChargerTransactionID"] in some:
        print(i['transaction']["ChargerTransactionID"])
        print(index)

for index, i in enumerate(all_transaction_info['TEzj7R']):
    if i['transaction']["ChargerTransactionID"] in some:
        print(i['transaction']["ChargerTransactionID"])
        print(index)

for index, i in enumerate(all_transaction_info['gwmF5L']):
    if i['transaction']["ChargerTransactionID"] in some:
        print(i['transaction']["ChargerTransactionID"])
        print(index)

for index, i in enumerate(all_transaction_info['HpqpIU']):
    if i['transaction']["ChargerTransactionID"] in some:
        print(i['transaction']["ChargerTransactionID"])
        print(index)

for index, i in enumerate(all_transaction_info['z4gi5Z']):
    if i['transaction']["ChargerTransactionID"] in some:
        print(i['transaction']["ChargerTransactionID"])
        print(index)


In [ ]:
print(all_transaction_info['z3CHWb'][103]["transaction"]["StartAt"], all_transaction_info['z3CHWb'][103]["transaction"]["EndAt"])
print(" ")
for event in all_transaction_info['z3CHWb'][103]["events"]:
    event_name = event['name']
    if (event_name == "Color" or event_name == "State"):
        event_name = event_name + "_" + event['value']
    if event_name in start_charging_events or event_name in end_charging_events:
        if event['name'] == "Color" or event['name'] == "State":
            print(event['name'], event['value'], event['event_time'])
        else:
            print(event['name'], event['event_time'])

print("----")
print(all_transaction_info['TEzj7R'][1270]["transaction"]["StartAt"], all_transaction_info['TEzj7R'][1270]["transaction"]["EndAt"])
print(" ")
for event in all_transaction_info['TEzj7R'][1270]["events"]:
    event_name = event['name']
    if (event_name == "Color" or event_name == "State"):
        event_name = event_name + "_" + event['value']
    if event_name in start_charging_events or event_name in end_charging_events:
        if event['name'] == "Color" or event['name'] == "State":
            print(event['name'], event['value'], event['event_time'])
        else:
            print(event['name'], event['event_time'])

In [ ]:
some2 = [
    "3ac4c8c3-9d8e-4031-b5b5-e08b786dfceb",
    "60091375-df63-4b52-a7a1-a4e200271e56",
    "2326e1a8-48e6-4217-9f4a-f4969ada3fa7",
    "9ea54c08-6e4f-452e-be25-81a0a15788db",
    "0487e52d-6f7a-4f1d-aa0c-8d3d5c39435f",
    "1234cadb-b07c-4957-9dbf-3d0557f61f1b",
    "90a1f17a-fd9c-45e5-871b-1f10ff0eefc8",
    "6aa2b795-7df2-4170-9a90-81578ef2d769",
    "5a131f19-a71f-4a7c-9fb6-e4c438d54f26",
    "ce741dc5-9253-4711-9070-19bfde19adc7",
    "c7e4a0ee-9398-4fb0-9e52-04b038dc76fa",
]

In [ ]:
transactions[[ True if np.any([i == element for element in some2]) else False  for i in transactions['ChargerTransactionID'] ]][['ChargerID', "StartAt", "EndAt", "EndResult", "InitialSoC", "EndSoC", "EnergyDelivered", "ExtraInfo", "ChargerTransactionID"]]


In [ ]:

all_transaction_info["gwmF5L"][121]['transaction']
all_transaction_info["gwmF5L"][121]['meter_values']

In [ ]:

all_transaction_info["HpqpIU"][597]['transaction']
# all_transaction_info["HpqpIU"][597]['meter_values']

In [ ]:
fig, axs = plot.subplots(2,1, sharex=True, figsize=(8,6))

prepare_subplots(axs)
start = pd.Timestamp("2025-11-30 23:26:15.745000+05:30")
end = pd.Timestamp("2025-12-01 00:11:51.614000+05:30")
for a in axs:
    a.tick_params(axis="x", rotation=45)
    # a.set_xticklabels([element.get_text()[3:] for element in a.get_xticklabels()])
    a.axvline(start, linestyle="--", color="g")
    a.axvline(end ,linestyle="--", color="g")
    a.set_xlim(start-pd.Timedelta("2m"), end+pd.Timedelta("2m"))
    

axs[0].axhline(0)
axs[0].axhline(89)
xs = [ pd.Timestamp("2025-12-01 00:06:16.461000+05:30"), pd.Timestamp("2025-12-01 00:11:52.550000+05:30 ")]
ys_soc = [85, 89]

axs[0].set_ylim(-5,100)
axs[0].plot(xs, ys_soc, marker="o")

axs[1].set_xticklabels([tick.get_text()[3:] for tick in a.get_xticklabels()])

plot.show()

In [ ]:
for index, i in enumerate(all_transaction_info['Kuj9Sn']):
    if i['transaction']['ChargerTransactionID'] == "5ceef79b-f7cd-4033-935c-49e7741714ac":
        print(index)

In [ ]:
transaction_info = all_transaction_info['Kuj9Sn'][5]['transaction']
meter_values = all_transaction_info['Kuj9Sn'][5]['meter_values']

fig, ax = plot.subplots()
timestamps = []
energies = []
for value in meter_values:
    timestamps.append(value['MeterValueTimeStamp'])
    RawData   = json.loads(value["RawData"])
    energies.append(float(RawData['EnergyDelivered']))

energies = np.array(energies)
energies = energies - energies[0]
ax.plot(timestamps, energies)
start = pd.Timestamp("2025-12-05 17:17:05.715000+05:30")
end = pd.Timestamp("2025-12-05 18:27:05.933000+05:30")
print(end-start)
print(energies[-1]/((end-start).total_seconds()/(60*60)))
ax.set_xlim(start-pd.Timedelta('10m'), end+ pd.Timedelta("20m"))
ax.axvline(pd.Timestamp("2025-12-05 18:27:05.933000+05:30"))
plot.show()


In [ ]:
specific_transaction = all_transaction_info['z4gi5Z'][57]

transaction_info = specific_transaction['transaction']
meter_values_info = specific_transaction['meter_values']
events_info = specific_transaction['events']
print(transaction_info)
print(len(meter_values_info))
print(len(events_info))